### Installing Semantic Link Labs

In [ ]:
%pip install semantic-link-labs

### Importing Modules

In [ ]:
import sempy_labs as sl
import json
import uuid

### Core Logic

In [ ]:
def update_or_create_udf(workspace_id, semantic_model_id, function_name, dax_text, lineage_tag=None):
    """
    Update or create a DAX UDF in a Fabric semantic model.
    
    Parameters:
    -----------
    workspace_id : str
        The Fabric workspace ID
    semantic_model_id : str
        The semantic model/dataset ID
    function_name : str
        Name of the function to update/create
    dax_text : str
        List of strings representing the DAX function expression
    lineage_tag : str, optional
        Lineage tag for the function (auto-generated if not provided)
    
    Returns:
    --------
    None
    """

    tmsl = sl.get_semantic_model_definition(
        dataset=semantic_model_id, 
        workspace=workspace_id,
        format='TMSL', 
        return_dataframe=False
    )

    if isinstance(tmsl, str):
        tmsl = json.loads(tmsl)


    if tmsl.get('compatibilityLevel', 0) < 1702:
        tmsl['compatibilityLevel'] = 1702

    old_model = tmsl['model']
    
    if 'functions' not in old_model:
        new_model = {}
        for key, value in old_model.items():
            if key == 'tables':
                new_model['functions'] = []
            new_model[key] = value
        tmsl['model'] = new_model

    functions = tmsl['model']['functions']

    existing_index = None
    for i, func in enumerate(functions):
        if func['name'] == function_name:
            existing_index = i
            break

    function_obj = {
        'name': function_name,
        'expression': [line.rstrip() for line in dax_text.split('\n')],
        'lineageTag': lineage_tag if lineage_tag else str(uuid.uuid4())
    }

    if existing_index is not None:
        if not lineage_tag:
            function_obj['lineageTag'] = functions[existing_index]['lineageTag']
        functions[existing_index] = function_obj
    else:
        functions.append(function_obj)

    sl.update_semantic_model_from_bim(
        dataset=semantic_model_id,
        workspace=workspace_id,
        bim_file=tmsl
    )

def delete_udf(workspace_id, semantic_model_id, function_name):
    """
    Delete a DAX UDF from a Fabric semantic model by its name.

    Parameters:
    -----------
    workspace_id : str
        The Fabric workspace ID
    semantic_model_id : str
        The semantic model/dataset ID
    function_name : str
        Name of the function to update/create
    
    Returns:
    --------
    None
    """
    tmsl = sl.get_semantic_model_definition(
        dataset=semantic_model_id, 
        workspace=workspace_id,
        format='TMSL', 
        return_dataframe=False
    )

    if isinstance(tmsl, str):
        tmsl = json.loads(tmsl)
    
    if 'functions' not in tmsl['model']:
        print(f"No functions block found in model {semantic_model_id}. Nothing to delete.")
        return
    
    functions = tmsl['model']['functions']

    initial_count = len(functions)
    functions[:] = [f for f in functions if f.get('name') != function_name]

    if len(functions) == initial_count:
        print(f"Function '{function_name}' not found. No changes made.")
        return
    else:
        print(f"Successfully removed '{function_name}'. Remaining functions: {len(functions)}")

    sl.update_semantic_model_from_bim(
        dataset=semantic_model_id,
        workspace=workspace_id,
        bim_file=tmsl
    )

def bulk_manage_udf(model_pairs, function_name, action='update', dax_text=None, lineage_tag=None):
    """
    Update, Create, or Delete a DAX UDF across multiple semantic models.
    
    Parameters:
    -----------
    model_pairs : list of tuples
        List of (workspace_id, semantic_model_id) tuples
    function_name : str
        Name of the function to manage
    dax_text : str, optional
        The DAX function as a string (required for update/create)
    action : str
        'update' to create/update, 'delete' to remove
    lineage_tag : str, optional
        Lineage tag for the function
    
    Returns:
    --------
    dict
        Dictionary with results: {'success': [...], 'failed': [...]}
    """
    results = {'success': [], 'failed': []}
    
    for workspace_id, semantic_model_id in model_pairs:
        try:
            print(f"\n{'='*60}")
            print(f"Action: {action.upper()} | Model: {semantic_model_id}")
            print(f"{'='*60}")
            
            if action.lower() == 'delete':
                delete_udf(
                    workspace_id=workspace_id,
                    semantic_model_id=semantic_model_id,
                    function_name=function_name
                )
            else:
                if not dax_text:
                    raise ValueError("dax_text is required for 'update' action.")
                    
                update_or_create_udf(
                    workspace_id=workspace_id,
                    semantic_model_id=semantic_model_id,
                    function_name=function_name,
                    dax_text=dax_text,
                    lineage_tag=lineage_tag
                )
            
            results['success'].append((workspace_id, semantic_model_id))
            print(f"✓ Success")
            
        except Exception as e:
            print(f"✗ Failed: {str(e)}")
            results['failed'].append((workspace_id, semantic_model_id, str(e)))
    
    print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
    print(f"Total Models: {len(model_pairs)}")
    print(f"Successfully {action}ed: {len(results['success'])}")
    print(f"Failed: {len(results['failed'])}")
    
    return results

### Example: Adding/Updating a UDF in a Single Semantic Model

In [ ]:
workspace_id = "0ff09209-3edf-49cf-9f74-91a0d588b031"
semantic_model_id = "7535e0df-63f3-4cb5-9e45-bf4ab75961cb"

function_name = "corporate.formattingString"
expression = """
(
	_measurename : STRING
) =>
	switch(
		_measurename,
		"Revenue Calcz", "$#,##0,.0K",
		"$#,##0"
	)
"""

update_or_create_udf(
    workspace_id=workspace_id,
    semantic_model_id=semantic_model_id,
    function_name=function_name,
    dax_text=expression
)

### Example: Deleting a UDF in a Single Semantic Model

In [ ]:
workspace_id = "0ff09209-3edf-49cf-9f74-91a0d588b031"
semantic_model_id = "7535e0df-63f3-4cb5-9e45-bf4ab75961cb"

function_name = "corporate.formattingString"

delete_udf(
    workspace_id=workspace_id,
    semantic_model_id=semantic_model_id,
    function_name=function_name
)

### Example: Bulk Inserting and Updating

In [ ]:
#(workspaceid, SemanticModel)
model_pairs = [
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "21cc43f4-e89c-4b5f-b97d-af6abb9cebf6"),
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "51a3dfd7-4d14-4376-a3fe-4cc898eb5caa"),
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "7535e0df-63f3-4cb5-9e45-bf4ab75961cb")
]

function_name = "corporate.formattingString2"
expression = """
(
	_measurename : STRING
) =>
	switch(
		_measurename,
		"Revenue Calcz", "$#,##0,.0K",
		"$#,##0"
	)
"""

results = bulk_manage_udf(
    model_pairs=model_pairs,
    function_name=function_name,
    dax_text=expression
)

### Example: Bulk Deleting

In [ ]:
#(workspaceid, SemanticModel)
model_pairs = [
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "21cc43f4-e89c-4b5f-b97d-af6abb9cebf6"),
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "51a3dfd7-4d14-4376-a3fe-4cc898eb5caa"),
    ("0ff09209-3edf-49cf-9f74-91a0d588b031", "7535e0df-63f3-4cb5-9e45-bf4ab75961cb")
]

function_name = "corporate.formattingString2"

results = bulk_manage_udf(
    model_pairs=model_pairs,
    function_name=function_name,
    action='delete'
)